In [0]:
source_dir="/Volumes/eccom_catalog/default/ecomm_data/customers_data/Source/"
archive_dir="/Volumes/eccom_catalog/default/ecomm_data/customers_data/Archive/"
customer_stage_table="eccom_catalog.default.customers_stage"
error_table="eccom_catalog.default.error_table"

In [0]:
#Read customers csv file
from pyspark.sql.functions import col,current_timestamp,current_date,lit,length,floor,months_between,datediff,when
from datetime import datetime
try:
    df_customer=spark.read.csv(source_dir, header=True, inferSchema=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")
    df_customer=df_customer.withColumn("processed_timestamp", current_timestamp())\
        .withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S")))\
            .withColumn("source_system", lit("ecommerce_customers"))
    print("Successfully read customers file")

except Exception as e:
    print(f"Failed to read customers file: {str(e)}")
    raise

In [0]:
#Apply data Quality checks
try:
    total_records=df_customer.count()
    total_null_ids=df_customer.filter(col("customer_id").isNull()).count()
    total_null_emails=df_customer.filter(col("email").isNull()).count()
    total_null_phone=df_customer.filter(col("phone").isNull()).count()
    total_future_birth_date=df_customer.filter(col("date_of_birth")>current_date()).count()
    total_invalid_emails=df_customer.filter(~((col("email").contains("@")) & (col("email").contains(".")))).count()
    total_invalid_phone=df_customer.filter(~((col("phone").contains("-")) & (length(col("phone"))==12))).count()
    print(f"Total records: {total_records}")
    print(f"Total null customer ids: {total_null_ids}")
    print(f"Total null emails: {total_null_emails}")
    print(f"Total null phones: {total_null_phone}")
    print(f"Total future birth date: {total_future_birth_date}")
    print(f"Total invalid emails: {total_invalid_emails}")
    print(f"Total invalid phones: {total_invalid_phone}")

    valid_records=df_customer.filter((col("customer_id").isNotNull()) & (col("email").isNotNull()) & (col("phone").isNotNull())
                                     & (col("date_of_birth")<=current_date()) & (col("email").contains("@")) & (col("email").contains("."))
                                     & (col("phone").contains("-")) & (length(col("phone"))==12))
    invalid_records=df_customer.filter((col("customer_id").isNull()) | (col("email").isNull()) | (col("phone").isNull())
                                       | (col("date_of_birth")>current_date()) | ~((col("email").contains("@")) & (col("email").contains("."))) | ~((col("phone").contains("-")) & (length(col("phone"))==12)))
    total_valid_records=valid_records.count()
    total_invalid_records=invalid_records.count()
    print(f"Total valid records: {total_valid_records}")
    print(f"Total invalid records: {total_invalid_records}")

except Exception as e:
    print(f"Failed in Data Quality checks: {str(e)}")
    raise    

In [0]:
#Data enrichment - calculate customer age segment
try:
    valid_records=valid_records.withColumn("age",floor(months_between(current_date(),col("date_of_birth"))/12))
    #Create age segment
    valid_records=valid_records.withColumn("age_segment", when(col("age")<25, "Gen Z")
                                           .when(col("age")<40, "Millennial")
                                           .when(col("age")<55, "Gen X")
                                           .otherwise("Boomer+"))
    #Calculate days since registration
    valid_records=valid_records.withColumn("days_since_registration",datediff(current_date(),col("registration_date")))
    #Calculate customer lifecycle stage 
    valid_records=valid_records.withColumn("lifecycle_stage", when(col("days_since_registration")<30, "New")
                                           .when(col("days_since_registration")<365, "Active")
                                           .otherwise("Established"))
    print("Data enrichment completed")

except Exception as e:
    print(f"Failed to Data enrichment: {str(e)}")
    raise

In [0]:
#Write in staging table
try:
    #write valid records to staging table
    valid_records.write.mode('overwrite').format('delta').saveAsTable(customer_stage_table)
    print(f"Successfully loaded {total_valid_records} valid customers to staging table")
    if total_invalid_records>0:
        invalid_records.withColumn('error_reason',lit('Data Quality validation fail')).withColumn('error_timestamp', current_timestamp())\
            .write.mode('append').format('delta').saveAsTable(error_table)
        print(f"Logged {total_invalid_records} invalid records to error table")
    else:
        print("No invalid records found")    

except Exception as e:
    print(f"Failed to write to staging table: {str(e)}")
    raise

In [0]:
#Archive processed files 
try:
    files=dbutils.fs.ls(source_dir)
    archive_count=0
    for file in files:
        if file.name.endswith('.csv'):
            source=file.path
            archive=archive_dir+file.name
            dbutils.fs.mv(source,archive)
            archive_count+=1
    print(f"Archived {archive_count} files")
except Exception as e:
    print(f"Failed to archive processed files: {str(e)}")
    raise

In [0]:
#Log processing summary
import json
try:
    processing_summary={
        "task": "customers_stage_load",
        "timestamp": datetime.now().isoformat(),
        "total_records": total_records,
        "valid_records": total_valid_records,
        "invalid_records": total_invalid_records,
        "archived_files": archive_count,
        "status": "SUCCESS" if total_invalid_records==0 else "SUCCESS_WITH_WARNINGS"
    }
    print(f"Processing Summary: {json.dumps(processing_summary)}")
    processing_log=spark.createDataFrame([processing_summary])
    processing_log.write.mode("append").format("delta").saveAsTable("eccom_catalog.default.processing_log")
    print("Successfully logged processing summary")

except Exception as e:
    print(f"Failed to log processing summary: {str(e)}")      